<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-11-cost-and-performance/lesson-11.1-ai-app-costs/notebooks/exercises-11.1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11.1 — AI App Costs
Netsetos GenAI Course v2.0 | Module 11: Cost & Performance

Cost drivers, $/request math, agent O(N²) growth, attribution, $50K→$15K walkthrough, India cross-over.


In [ ]:
print('Module 11: Cost & Performance')
print('Lesson 11.1: AI App Costs')


## Ex 1: 2026 Pricing Tiers ($/MTok)


In [ ]:
print('Pricing tiers (illustrative 2026 rates):')
for tier, models, in_price, out_price in [
    ('Nano/Flash','Gemini 2.5 Flash, GPT-5 Nano, Claude Haiku', 0.10, 0.40),
    ('Mid',       'GPT-4o, Sonnet 4.6',                         3.00, 15.00),
    ('Frontier',  'Opus 4.7, GPT-5.3 Pro',                      15.00, 75.00),
    ('Self-host', 'Llama-3.1-70B on H100 (vLLM)',                0.0,   0.0),
]:
    print(f'  {tier:10s} | in ${in_price:6.2f}/M  out ${out_price:6.2f}/M | {models}')
print()
print('Output is 3-5x input. Frontier is ~75x Nano. Choose tier per task complexity.')


## Ex 2: $/Request Math


In [ ]:
print('$/request for a typical RAG app (Sonnet 4.6):')
in_price, out_price = 3.0, 15.0
for in_tok, out_tok, label in [
    ( 500,  200, 'simple Q&A'),
    (3000,  500, 'RAG (5 chunks)'),
    (8000, 1500, 'long context + reasoning'),
    (5000, 5000, 'agent step (heavy output)'),
]:
    cost = (in_tok*in_price + out_tok*out_price) / 1_000_000
    print(f'  {label:24s} in={in_tok:>5}  out={out_tok:>4} -> ${cost:.4f}/call')


## Ex 3: Agent O(N²) Cost Compounding


In [ ]:
S, u, r = 1000, 2500, 500    # system, new input/iter, output/iter (tokens)
in_price, out_price = 3.0, 15.0
print('Agent loop cost growth (Sonnet pricing):')
single = (S + u) * in_price / 1_000_000 + r * out_price / 1_000_000
for N in [1, 3, 5, 10, 15, 20]:
    in_tot = N*S + u*N*(N+1)//2
    out_tot = r*N*(N-1)//2 + r*N
    cost = (in_tot*in_price + out_tot*out_price) / 1_000_000
    mult = cost / single
    print(f'  N={N:2d} | input={in_tot:>7,} out={out_tot:>5,} -> ${cost:.3f} ({mult:>4.0f}x single)')
print()
print('Mitigations: prompt cache (90% off prefix), compaction, sliding window, max_turns=15-25.')


## Ex 4: Cost-Cut Recipes Stacked


In [ ]:
monthly_baseline = 50_000  # $/mo
print(f'Starting monthly bill: ${monthly_baseline:,}')
print()
print('Stacked optimizations (real customer-service bot example):')
current = monthly_baseline
for step, savings_pct in [
    ('Semantic cache (31% hit)',          0.32),
    ('Anthropic prompt cache on system',  0.30),
    ('Cost router 70/22/8 (Flash/Haiku/Sonnet)', 0.31),
    ('Output token reduction (compact prompts)', 0.10),
]:
    new = current * (1 - savings_pct)
    print(f'  {step:48s} ${current:>7,.0f} -> ${new:>7,.0f}  (-{savings_pct*100:.0f}%)')
    current = new
print()
print(f'Total cut: ${monthly_baseline:,} -> ${current:,.0f}  ({(1-current/monthly_baseline)*100:.0f}% reduction)')


## Ex 5: Hidden Costs Checklist


In [ ]:
print('Hidden costs people forget:')
for item, impact in [
    ('Embedding re-runs on doc updates',     '$0.02/MTok x corpus size x update freq'),
    ('Eval-on-prod judge spend',             '~$10/day at 1% sample x 100k calls'),
    ('Failed + retried calls',               '2-3x base cost on bad days'),
    ('Long-context attention overhead',      'quadratic compute even within cache'),
    ('Forex (USD billing)',                  '3-5% drag if not billed INR'),
    ('Egress to provider',                   'small but nonzero on K8s'),
    ('Vector DB storage + query',            '$0.04-0.10/M vectors/mo'),
]:
    print(f'  {item:36s}: {impact}')


## Ex 6: India Cross-Over Math (API vs Self-Host)


In [ ]:
print('When does self-hosting beat the API? (E2E spot pricing):')
print()
for model, gpu_rate_inr_hr, gpus, api_cost_per_call in [
    ('Llama-3.1-8B',  150,  1,  0.0008),
    ('Llama-3.1-70B', 600,  4,  0.0050),
    ('Mixtral-8x22B', 600,  4,  0.0040),
]:
    monthly_gpu = gpu_rate_inr_hr * 24 * 30 * gpus / 88  # INR -> USD approx
    breakeven = monthly_gpu / api_cost_per_call
    print(f'  {model:18s} | gpu ${monthly_gpu:>6.0f}/mo | breakeven ~{breakeven:>7,.0f} req/mo')
print()
print('Below cross-over: API + cache is cheaper. Above: self-host wins (add 20% ops overhead).')


## Ex 7: Cost Attribution Tagging


In [ ]:
print('Tag every LLM call with:')
for field, why in [
    ('tenant_id',       'aggregate spend per customer'),
    ('feature',         'know which feature drives cost'),
    ('prompt_version',  'attribute regressions to prompt changes'),
    ('virtual_key',     'enforce budget cap per env/team'),
    ('user_segment',    'free/paid/enterprise pricing analysis'),
    ('trace_id',        'link to OTel + Langfuse for drill-down'),
]:
    print(f'  {field:18s}: {why}')
print()
print('Without tags you cannot optimize what you cannot see.')


---
## Recap
Cost = tokens (80%) + GPU rental + observability. Output is 3-5x input.
Agent loops cost O(N²); cap turns + cache + compact.
Stack 4 cost cuts: cache + prompt cache + cost router + compact prompts = 70%+ reduction.
Tag everything (tenant + feature + prompt_version) or you can't optimize.
India: E2E + IndiaAI + $600K credits + INR billing. Self-host above ~300K req/mo for 8B.
